## Heterogeneous effects: metalearners, causal forest and checking a CATE estimate

Practice for the module [“Heterogeneous Effects”](https://ai-interpretability.school/learn/5/0/0) of AI Interpretability School, lessons “Metalearners: S, T and X” and “The causal forest and honest splitting”.

In the module we went through why the individual effect $\tau_i = Y_i(1) - Y_i(0)$ is never observed, how metalearners and the causal forest estimate the conditional average effect $\tau(x)$, and why an estimate can only be checked on groups of objects. In this notebook all of that is done on synthetic data where the true $\tau(x)$ is known:

1. S-, T- and X-learner in EconML and the scatter “true effect versus estimated”;
2. the causal forest `CausalForestDML` with confidence intervals;
3. calibration by quantiles of the predicted effect;
4. the uplift curve, the Qini curve and the Qini coefficient;
5. the DR-learner by hand: a doubly robust pseudo-outcome with cross-fitting;
6. where metalearners break: the S-learner with lasso and the DR-learner under extreme propensities.

On real data a comparison with the true effect is impossible, so steps 3 and 4 are the checks that remain when working with real data, and steps 1 and 2 show how well they agree with the truth.

In [ ]:
!pip install -q econml==0.17.0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso
from sklearn.model_selection import KFold
from econml.metalearners import SLearner, TLearner, XLearner
from econml.dml import CausalForestDML
from econml.dr import DRLearner

### Data

A randomized experiment: the treatment is given to a random fifth of the objects, the probability $0.2$ does not depend on the features. There are five features, $X \sim \mathcal N(0, I)$.

- The effect depends only on the first feature: $\tau(x) = 1 + 1.5\,x_0$. For objects with $x_0 < -2/3$ the effect is negative: the treatment harms them.
- The outcome without treatment depends on the second and third features: $\mu_0(x) = 2x_1 + x_2^2$. These features explain the outcome but do not distinguish the effect — like age in the example from the lesson “The causal forest and honest splitting”.
- The observed outcome: $Y = \mu_0(X) + T\,\tau(X) + \varepsilon$, $\varepsilon \sim \mathcal N(0, 1)$.

Half of the sample goes to training, half to the held-out check.

In [ ]:
rng = np.random.default_rng(0)
n, d = 4000, 5
X = rng.normal(size=(n, d))
tau = 1 + 1.5 * X[:, 0]                       # true CATE
T = rng.binomial(1, 0.2, size=n)              # treatment: a random fifth
mu0 = 2 * X[:, 1] + X[:, 2] ** 2              # outcome without treatment
Y = mu0 + T * tau + rng.normal(size=n)

idx = rng.permutation(n)
tr, te = idx[: n // 2], idx[n // 2 :]
print('train:', len(tr), '|', 'held-out:', len(te), '|', 'treated share:', round(T.mean(), 3))
print('share of objects with a negative effect:', round((tau < 0).mean(), 3))

**Task 1. Fit S-, T- and X-learner from `econml.metalearners` on the training half. Inside all schemes use `GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=0)`; the propensity model of X-learner is `LogisticRegression()`. Compute the CATE estimates on the held-out half (`tau_s`, `tau_t`, `tau_x`) and the RMSE of each against the true `tau[te]`. Which scheme is closest to the true effect?**

Write the RMSE as a function `rmse(est)` and the model inside the schemes as a function `gb()`: you will need them later.

In [ ]:
# Your code here

The scatter “true effect versus estimated”: the true $\tau(x)$ on the X axis, the scheme's estimate on the Y axis, the red line is $y = x$. The closer the cloud of points is to the line, the more precisely the scheme recovers the effect on individual objects. Such a picture can only be drawn on synthetic data.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharex=True, sharey=True)
for ax, (name, est) in zip(axes, (('S-learner', tau_s), ('T-learner', tau_t), ('X-learner', tau_x))):
    ax.scatter(tau[te], est, s=6, alpha=0.35, color='#3b6ea5')
    lim = [tau[te].min(), tau[te].max()]
    ax.plot(lim, lim, color='#c0392b', lw=1.5, label='y = x')
    ax.set_title(f'{name}: RMSE = {rmse(est):.2f}')
    ax.set_xlabel('True effect τ(x)')
    ax.legend(loc='upper left')
axes[0].set_ylabel('Estimated effect τ̂(x)')
fig.suptitle('True versus estimated CATE on the held-out half')
fig.tight_layout()
fig.savefig('cate_true_vs_estimated.png', dpi=150, bbox_inches='tight')
plt.show()

**Task 2. Fit the causal forest `CausalForestDML` on the training half: `model_y=gb()`, `model_t=GradientBoostingClassifier(random_state=0)`, `discrete_treatment=True`, `n_estimators=1000`, `min_samples_leaf=10`, `random_state=0`; honest splitting is on by default (`honest=True`). Compute on the held-out half the estimate `tau_cf`, its RMSE and the 95% confidence intervals `effect_interval(..., alpha=0.05)`. What share of held-out objects has the true effect inside its interval?**

In [ ]:
# Your code here

### Calibration by quantiles of the predicted effect

Next come the checks that do not require knowing the true effect. Sort the held-out objects by the predicted effect and split them into four groups of a quarter each. Within each group compute the actual difference in mean outcomes between treated and control objects: the experiment is randomized, so this difference estimates the average effect in the group. If the model ranks the effect meaningfully, the actual difference grows together with the predicted one.

**Task 3. Split the held-out half into quartiles by `tau_cf` (`pd.qcut(tau_cf, 4, labels=[1, 2, 3, 4])`). For each quartile print the mean predicted effect, the actual difference in mean outcomes of treated and control objects and, for comparison, the mean true effect. Does the actual difference grow from the first quartile to the fourth?**

In [ ]:
# Your code here

### The uplift curve and the Qini curve

Calibration by quartiles gives four points. The uplift curve does the same for any share of objects: sort the objects by $\hat\tau(x)$ in descending order, take the top share $\varphi$ and compute what gain in the outcome per object of the whole sample treating only this share would give: $(\bar Y_{T=1}(\varphi) - \bar Y_{T=0}(\varphi))\cdot\varphi$. A random order gives a straight line from zero to the difference in mean outcomes of treated and control objects over the whole sample.

The Qini curve is built the same way but counts the gain through sums: $Q(\varphi) = \bigl(S_{T=1}(\varphi) - S_{T=0}(\varphi)\cdot N_{T=1}(\varphi)/N_{T=0}(\varphi)\bigr)/N_{T=1}$, where $S$ is the sum of outcomes and $N$ the number of objects of the group within the top share $\varphi$. **The Qini coefficient** is the area between the Qini curve and the random-order line: the larger it is, the better the model separates objects with a large effect from objects with a small one.

**Task 4. Write a function `qini_curve(tau_hat, y, t)` that returns the share $\varphi$ (from 0 to 1 in steps of one object), the values $Q(\varphi)$ and the Qini coefficient — the area $\int_0^1 \bigl(Q(\varphi) - \varphi\,Q(1)\bigr)\,d\varphi$ by the trapezoidal rule (`np.trapezoid`). Compute the coefficient on the held-out half for the causal forest, for X-learner and for sorting by the true effect `tau[te]`.**

In [ ]:
# Your code here

Left: the uplift curves of the causal forest, X-learner and sorting by the true effect; the dotted line is a random order. Right: the Qini curve of the causal forest; the shaded area between it and the random-order line is the Qini coefficient.

In [ ]:
def uplift_curve(tau_hat, y, t):
    order = np.argsort(-tau_hat)
    y, t = y[order], t[order]
    n_t, n_c = np.cumsum(t), np.cumsum(1 - t)
    mean_t = np.divide(np.cumsum(y * t), n_t, out=np.zeros(len(y)), where=n_t > 0)
    mean_c = np.divide(np.cumsum(y * (1 - t)), n_c, out=np.zeros(len(y)), where=n_c > 0)
    k = np.arange(1, len(y) + 1)
    gain = np.where((n_t > 0) & (n_c > 0), (mean_t - mean_c) * k / len(y), 0.0)
    return np.r_[0, k / len(y)], np.r_[0, gain]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6))
for name, est, color in (('Causal forest', tau_cf, '#3b6ea5'), ('X-learner', tau_x, '#e67e22'),
                         ('Sorted by true effect', tau[te], '#7f8c8d')):
    phi, g = uplift_curve(est, y_te, t_te)
    ax1.plot(phi, g, color=color, lw=1.8, ls='--' if name.startswith('Sorted') else '-', label=name)
ax1.plot([0, 1], [0, g[-1]], color='black', lw=1, ls=':', label='Random order')
ax1.set_title('Uplift curve')
ax1.set_xlabel('Share of objects treated, sorted by predicted effect')
ax1.set_ylabel('Cumulative gain per object')
ax1.legend()

phi, q, coef = qini_curve(tau_cf, y_te, t_te)
ax2.plot(phi, q, color='#3b6ea5', lw=1.8, label='Causal forest')
ax2.plot([0, 1], [0, q[-1]], color='black', lw=1, ls=':', label='Random order')
ax2.fill_between(phi, phi * q[-1], q, color='#3b6ea5', alpha=0.2, label=f'Qini coefficient = {coef:.3f}')
ax2.set_title('Qini curve')
ax2.set_xlabel('Share of objects treated, sorted by predicted effect')
ax2.set_ylabel('Qini: incremental gain per treated object')
ax2.legend()
fig.tight_layout()
fig.savefig('uplift_qini_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### DR-learner: a doubly robust pseudo-outcome

The X-learner fills in the missing potential outcome with a model and mixes two effect models by the propensity. The DR-learner ([Kennedy, 2023](https://arxiv.org/abs/2004.14497)) builds one number per object — a pseudo-outcome whose conditional mean equals the effect when the auxiliary models are correct — and regresses it on the features:

$$\varphi_i = \hat\mu_1(X_i) - \hat\mu_0(X_i) + \frac{T_i\,(Y_i - \hat\mu_1(X_i))}{\hat e(X_i)} - \frac{(1-T_i)\,(Y_i - \hat\mu_0(X_i))}{1-\hat e(X_i)}.$$

Here $\hat\mu_1$ and $\hat\mu_0$ are outcome models fitted on the treated and on the controls, and $\hat e$ is the propensity model. The pseudo-outcome of an object is computed by models that have not seen this object (cross-fitting): the sample is split into five folds, and for the objects of each fold the auxiliary models are fitted on the other four.

The experiment in this notebook is randomized, and the true propensity is $0.2$ for everyone; we still fit a propensity model, as one would have to on observational data.

**Task 5. Write a function `dr_pseudo(Xa, Ta, Ya, prop_model, clip=None)` that returns the pseudo-outcome $\varphi_i$ and the propensity $\hat e(X_i)$ for every object of the sample `(Xa, Ta, Ya)`. The split is `KFold(5, shuffle=True, random_state=0)`; in each fold the outcome models `gb()` are fitted separately on the treated and on the control objects of the other folds, and the propensity model `prop_model()` on all objects of the other folds; if `clip` is given, the propensity is clipped to $[\text{clip}, 1-\text{clip}]$. Compute the pseudo-outcome on the training half with `LogisticRegression` and fit two effect models on it: `LinearRegression()` and `gb()`. Print the RMSE of both estimates on the held-out half (`tau_dr_lin`, `tau_dr_gb`) and, for comparison, the RMSE of the X-learner and of `econml.dr.DRLearner` with `model_regression=gb()`, `model_propensity=LogisticRegression()`, `model_final=gb()`, `cv=5`, `random_state=0`.**

In [ ]:
# Your code here

The pseudo-outcome is noisy: for a treated object the outcome residual is divided by a propensity of about $0.2$, that is, multiplied by about five. A linear effect model averages this noise out and, besides, belongs to the same class as the true $\tau(x)=1+1.5\,x_0$, so its error is small. Boosting of depth 3 fits the noise of the pseudo-outcome and errs more than the X-learner — the same conclusion as in the notebook “R-learner”: the effect model needs a simple form or strong regularization.

`econml.dr.DRLearner` is built differently from Kennedy's scheme: instead of two outcome models by group it fits one model `model_regression` on the features and the treatment together (like the S-learner) and clips the propensity from below and above at `min_propensity=1e-6`. That is why the library's number differs from ours.

### Where it breaks: the S-learner with lasso

The lesson “Metalearners: S, T and X” warns: for the S-learner the treatment is one feature among many, and regularization can drive its coefficient to zero. Let us check it on a linear S-learner: lasso on the features $[X, T]$. In a model without interactions there is one effect estimate for everyone — the coefficient of $T$.

**Task 6. For each `alpha` in `(0.01, 0.03, 0.1, 0.2, 0.3, 0.5, 1.0)` fit `Lasso(alpha=alpha)` on the features `np.c_[X[tr], T[tr]]` and the outcome `Y[tr]` and print a table: the coefficient of $T$ (the S-learner's effect estimate), the coefficient of $x_1$ and the number of nonzero coefficients of the features $x_0,\dots,x_4$. At which smallest `alpha` of the grid does the effect estimate become exactly zero, and what happens to the coefficient of $x_1$ at that point?**

In [ ]:
# Your code here

**Your answer:**

### Where it breaks: the DR-learner under extreme propensities

The weak spot of the DR-learner is the same as that of IPW: division by $\hat e(x)$ and $1-\hat e(x)$. Take observational data where the treatment depends on the same feature as the effect: the propensity is $e(x)=1/(1+e^{-3x_0})$, and for objects with large $|x_0|$ it is close to 0 or 1. The effect and the outcome without treatment are the same as above; 2000 objects for training, 2000 for checking.

In [ ]:
rc = np.random.default_rng(1)
Xc = rc.normal(size=(n, d))
tau_c = 1 + 1.5 * Xc[:, 0]
Tc = rc.binomial(1, 1 / (1 + np.exp(-3 * Xc[:, 0])))       # propensity grows with x0
Yc = 2 * Xc[:, 1] + Xc[:, 2] ** 2 + Tc * tau_c + rc.normal(size=n)
tr_c, te_c = np.arange(n // 2), np.arange(n // 2, n)
rmse_c = lambda est: np.sqrt(np.mean((est - tau_c[te_c]) ** 2))
print('treated share:', round(Tc.mean(), 3))

**Task 7. On the training part `tr_c` compute the pseudo-outcome with `dr_pseudo` and `LogisticRegression` twice: without clipping and with `clip=0.05`. For each variant print the share of objects whose propensity before clipping lies outside $[0.05, 0.95]$, the largest $|\varphi_i|$ and the RMSE of the DR-learner with the effect model `gb()` on `te_c`. For comparison print the RMSE of the X-learner with `gb()` and `LogisticRegression()`. What does clipping do, and at what price?**

In [ ]:
# Your code here

**Your answer:**

**Conclusions**

- On synthetic data we see what is unavailable when working with real data: how far each estimate $\hat\tau(x)$ is from the true effect on an individual object. Compare the RMSE of the schemes with what the lesson “Metalearners: S, T and X” says about group imbalance: here the treatment was given to a fifth of the sample.
- The confidence intervals of the causal forest are asymptotic. Compare the share of objects whose true effect fell inside the interval with the nominal 95%.
- Calibration by quantiles, the uplift curve and the Qini coefficient do not require knowing the true effect, but they check only the ranking by groups, not the precision on an object.
- The features $x_1$ and $x_2$ determine the outcome but not the effect: an outcome model would rank them as important, while for the decision “whom to treat” they give nothing.
- The DR-learner reduces CATE estimation to a regression of a pseudo-outcome, but the pseudo-outcome is noisy: the effect model needs a simple form, and the propensity has to stay away from 0 and 1.
- An S-learner with lasso can zero out the effect entirely while keeping the features that explain the outcome better.

Sources:
1. [Künzel et al., 2019. Metalearners for estimating heterogeneous treatment effects using machine learning](https://arxiv.org/abs/1706.03461)
2. [Wager and Athey, 2018. Estimation and inference of heterogeneous treatment effects using random forests](https://arxiv.org/abs/1510.04342)
3. [Kennedy, 2023. Towards optimal doubly robust estimation of heterogeneous causal effects](https://arxiv.org/abs/2004.14497)
4. [EconML](https://github.com/py-why/EconML)